# 08 — Gate AB-2 analysis: verdict rule v2, E14-analog, the D-AB5 v2 nesting test, S4 pilot (Python)

Runs AFTER 06 + 07. Kernel `y2y-geo`. Zero solves. Everything pre-registered:

- **Verdict rule v2** (parent hash `v2_8db80fed1c702638`, asserted unmodified): D = maxHam/(2·m_disc),
  C = f=1 core share, over anchor + 50 members, discretionary cells; PLATEAU-RICH iff D ≥ 0.10 ∧ C ≤ 0.90;
  NEAR-UNIQUE iff D < 0.02 ∨ C > 0.98. → **H-AB4** at level A (and B, reported).
- **E14-analog**: share of aggregate-band members carrying ≥1 block below 0.95× the anchor's block
  capture (block capture = Σ member-feature captured fractions, the floor formula).
- **Nesting test (D-AB5 v2, frozen):** N = |core_B ∩ core_A| / |core_B| on the guarded frequent tiers
  (F ≥ 0.70, discretionary). N ≥ 0.80 ⇒ level A primary; else level B primary.
- **S4 pilot** (M4.3): m_soc capture at the kink? θ-tail mass capture ≥ 0.75 (m_soc only; biomass reported).
- f(g) core erosion at A; T1-analog block captures (intended shares beside realized captures).

In [1]:
# ---- bootstrap -----------------------------------------------------------------------------------------
import hashlib, importlib, json, pathlib, sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import rasterio

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec
for _m in (config, lc, ec):
    importlib.reload(_m)

HERE = ROOT / "analyses" / "alberta_prioritization"
SPEC, FIGS, RUNS = HERE / "spec", HERE / "figures", HERE / "runs" / "ab_l"
AB = config.AB_HANDOFF_DIR
SC = json.loads((SPEC / "scenarios_ab_v1.json").read_text())
LV = json.loads((SPEC / "ab_budget_levels_v1.json").read_text())
EXTENT = json.loads((SPEC / "ab_extent_v1.json").read_text())
BLOCKS = SC["_meta"]["blocks"]
REF = "s0_ssp585_theta5"
TH = SC["_meta"]["s4_ladder"]["chosen_theta"]
S4_ID = f"s4_ssp585_theta{str(TH).rstrip('0').rstrip('.')}"

RULE = ("v2: D=maxHam/(2*m_disc), C=all-selected/m_disc, over anchor+k at g, discretionary; "
        "PLATEAU-RICH iff D>=0.10 and C<=0.90; NEAR-UNIQUE iff D<0.02 or C>0.98; else INTERMEDIATE")
RULE_HASH = "v2_" + hashlib.sha256(RULE.encode()).hexdigest()[:16]
assert RULE_HASH == "v2_8db80fed1c702638", f"verdict rule text drifted from the parent's frozen hash: {RULE_HASH}"
NEST_T = LV["_meta"]["nesting_threshold"]
FREQ = 0.70

pu = lc.pu_mask(AB)
with rasterio.open(AB / "mask_protected_areas.tif") as s:
    locked = (s.read(1) == 1) & pu
disc_v = (~locked)[pu]                           # discretionary mask over PU columns
FEAT = {f: np.nan_to_num(lc._read(AB / f"{f}.tif")[pu], nan=0.0) for b in BLOCKS.values() for f in b}
SHARE = {f: v / v.sum() for f, v in FEAT.items()}
print(f"rule {RULE_HASH} OK | nesting threshold {NEST_T} | PU {pu.sum():,} | discretionary {disc_v.sum():,} | S4 id {S4_ID}")

def sols(level, tag):
    cd = RUNS / level / REF
    A = ec.read_selections(cd / "anchor.tif", pu)
    M = ec.read_selections(cd / f"mga_{tag}.tif", pu)
    return np.vstack([A, M])                    # (1 + k) x n_pu, anchor first
def block_capture(x):
    return {b: float(sum(SHARE[f][x].sum() for f in fs)) for b, fs in BLOCKS.items()}

rule v2_8db80fed1c702638 OK | nesting threshold 0.8 | PU 85,133 | discretionary 57,161 | S4 id s4_ssp585_theta2


## A — verdict rule v2 at both levels (H-AB4) + E14-analog

In [2]:
# ---- D, C, verdict; E14-analog; frequent tiers under both semantics ------------------------------------
rows, F = [], {}
for level in ("A", "B"):
    for tag in ("g05", "guard_g05"):
        S = sols(level, tag)
        Sd = S[:, disc_v].astype(np.float32)
        m_disc = int(Sd[0].sum())
        G = Sd @ Sd.T
        n = Sd.sum(1)
        ham = n[:, None] + n[None, :] - 2 * G
        D = float(ham.max() / (2 * m_disc))
        f = Sd.mean(0)
        C = float((f >= 1 - 1e-9).sum() / m_disc)
        verdict = ("PLATEAU-RICH" if (D >= 0.10 and C <= 0.90) else
                   "NEAR-UNIQUE" if (D < 0.02 or C > 0.98) else "INTERMEDIATE")
        F[(level, tag)] = f
        # E14-analog on the aggregate band: members sacrificing >= 1 block below 0.95 x anchor
        ab = block_capture(S[0]); sac = 0
        for i in range(1, S.shape[0]):
            cb = block_capture(S[i])
            if any(cb[b] < 0.95 * ab[b] - 1e-9 for b in BLOCKS): sac += 1
        rows.append(dict(level=level, semantics="guarded" if "guard" in tag else "aggregate", m_disc=m_disc,
                         D=round(D, 4), C=round(C, 4), verdict=verdict,
                         frequent_km2=int((f >= FREQ).sum()), always_km2=int((f >= 0.95).sum()),
                         union_km2=int((f > 0).sum()), members_sacrificing_a_block=sac, k=S.shape[0] - 1))
V = pd.DataFrame(rows)
print(V.to_string(index=False))
hab4 = V[(V.level == "A") & (V.semantics == "aggregate")].iloc[0]
print(f"\nH-AB4 (level A, aggregate band): D {hab4.D} (parent 0.953) | C {hab4.C} (parent 0.020) -> {hab4.verdict}; "
      f"direction registered: D below / C above the parent -> {'CONFIRMED' if (hab4.D < 0.953 and hab4.C > 0.020) else 'NOT as registered'}")
e14 = V[V.semantics == "aggregate"]
print("E14-analog: " + " | ".join(f"level {r.level}: {r.members_sacrificing_a_block}/{r.k} members drop a block below 0.95x anchor" for r in e14.itertuples()))

level semantics  m_disc      D   C      verdict  frequent_km2  always_km2  union_km2  members_sacrificing_a_block  k
    A aggregate   10083 0.9999 0.0 PLATEAU-RICH             2           0      57161                            1 50
    A   guarded   10083 0.9999 0.0 PLATEAU-RICH             2           0      57161                            1 50
    B aggregate    5042 1.0000 0.0 PLATEAU-RICH             0           0      57161                            0 50
    B   guarded    5042 1.0000 0.0 PLATEAU-RICH             0           0      57161                            0 50

H-AB4 (level A, aggregate band): D 0.9999 (parent 0.953) | C 0.0 (parent 0.020) -> PLATEAU-RICH; direction registered: D below / C above the parent -> NOT as registered
E14-analog: level A: 1/50 members drop a block below 0.95x anchor | level B: 0/50 members drop a block below 0.95x anchor


## B — the D-AB5 v2 nesting test (frozen rule) + f(g) core erosion at A

In [3]:
# ---- nesting: guarded frequent tiers (primary), aggregate tiers (reported) ------------------------------
def core(level, tag):
    return F[(level, tag)] >= FREQ
res = {}
for tag, lab in (("guard_g05", "guarded"), ("g05", "aggregate")):
    cA, cB = core("A", tag), core("B", tag)
    N = float((cA & cB).sum() / max(cB.sum(), 1))
    res[lab] = dict(N=round(N, 4), core_A_km2=int(cA.sum()), core_B_km2=int(cB.sum()), overlap_km2=int((cA & cB).sum()))
    print(f"{lab:<10} core A {cA.sum():6,} km² | core B {cB.sum():6,} km² | B inside A {int((cA & cB).sum()):6,} -> N = {N:.3f}")
N = res["guarded"]["N"]
PRIMARY = "A" if N >= NEST_T else "B"
VACUOUS = res["guarded"]["core_B_km2"] == 0 or res["guarded"]["core_A_km2"] == 0
print(f"\nNESTING VERDICT (guarded, frozen threshold {NEST_T}): N = {N:.3f} -> "
      f"{'NESTED: level A primary (tiers of the wide envelope deliver the tight-envelope answer)' if PRIMARY == 'A' else 'NOT NESTED: level B primary for the applied deliverable; A stays the methods mirror'}")
if VACUOUS:
    print("   ** VACUOUS at g=5%: at least one guarded core is EMPTY -- there is nothing to nest. The frozen rule fires by "
          "arithmetic, not by evidence; the decision goes to chat (M9). Nesting at tighter bands is reported below.")
# M9: the same statistic at every band available at BOTH levels (the parent's sanctioned tighter-g dial)
by_band = {}
for tag in ("g02", "g05"):
    if all((RUNS / lvl / REF / f"mga_{tag}.tif").exists() for lvl in ("A", "B")):
        fa, fb = sols("A", tag)[:, disc_v].mean(0), sols("B", tag)[:, disc_v].mean(0)
        cA, cB = fa >= FREQ, fb >= FREQ
        by_band[tag] = dict(N=round(float((cA & cB).sum() / max(cB.sum(), 1)), 4), core_A_km2=int(cA.sum()), core_B_km2=int(cB.sum()),
                            always_A=int((fa >= 1 - 1e-9).sum()), always_B=int((fb >= 1 - 1e-9).sum()))
        print(f"   band {tag}: core A {cA.sum():,} km² | core B {cB.sum():,} km² | B inside A {int((cA & cB).sum()):,} -> N = {by_band[tag]['N']:.3f}")
res["by_band"] = by_band; res["vacuous_at_g05"] = bool(VACUOUS)
# D-AB10 (ratified): when the 5% test is vacuous, the frozen rule (threshold unchanged) is applied at the tightest
# band where BOTH levels have non-empty cores; if none, B is primary by the realism argument.
if VACUOUS:
    usable = [t for t in ("g02", "g05") if t in by_band and by_band[t]["core_A_km2"] > 0 and by_band[t]["core_B_km2"] > 0]
    if usable:
        PRIMARY = "A" if by_band[usable[0]]["N"] >= NEST_T else "B"
        res["decided_at_band"] = usable[0]
        print(f"   D-AB10: decided at band {usable[0]} (N = {by_band[usable[0]]['N']:.3f} vs {NEST_T}) -> level {PRIMARY} primary")
    else:
        PRIMARY = "B"; res["decided_at_band"] = None
        print("   D-AB10: no band with non-empty cores at both levels -> level B primary (realism default)")
else:
    res["decided_at_band"] = "g05"

# f(g) core erosion at level A (E4 mirror), if the grid ran
fg = {}
for g in (0.02, 0.05, 0.10):
    tag = f"g{int(round(100*g)):02d}"
    if (RUNS / "A" / REF / f"mga_{tag}.tif").exists():
        S = sols("A", tag); f = S[:, disc_v].mean(0)
        fg[g] = dict(always=int((f >= 1 - 1e-9).sum()), frequent=int((f >= FREQ).sum()), union=int((f > 0).sum()))
if fg:
    print("\nf(g) at level A (discretionary cells): " + " | ".join(
        f"g={g:.0%}: core {d['always']:,} / frequent {d['frequent']:,} / union {d['union']:,}" for g, d in fg.items()))

guarded    core A      2 km² | core B      0 km² | B inside A      0 -> N = 0.000
aggregate  core A      2 km² | core B      0 km² | B inside A      0 -> N = 0.000

NESTING VERDICT (guarded, frozen threshold 0.8): N = 0.000 -> NOT NESTED: level B primary for the applied deliverable; A stays the methods mirror
   ** VACUOUS at g=5%: at least one guarded core is EMPTY -- there is nothing to nest. The frozen rule fires by arithmetic, not by evidence; the decision goes to chat (M9). Nesting at tighter bands is reported below.
   band g02: core A 1,488 km² | core B 404 km² | B inside A 404 -> N = 1.000
   band g05: core A 2 km² | core B 0 km² | B inside A 0 -> N = 0.000

f(g) at level A (discretionary cells): g=2%: core 876 / frequent 1,488 / union 50,824 | g=5%: core 0 / frequent 2 / union 57,161 | g=10%: core 0 / frequent 0 / union 57,161


## C — S4 pilot (M4.3) + T1-analog block captures

In [4]:
# ---- S4 pilot: binding at the kink + theta-tail mass capture (m_soc only; biomass reported) -----------------
TOL = 0.005
def rep(level, fid, art="anchor"):
    return pd.read_csv(RUNS / level / fid / art / "portfolio_representation.csv").set_index("feature")["relative_held"]
theta = config.AUDIT["theta"]
soc = lc._read(AB / "irrecoverable_carbon_m_soc.tif"); bio = lc._read(AB / "irrecoverable_carbon_biomass.tif")
def tail_capture(x_full, arr):
    cut = theta * float(np.nanmean(arr[pu])); tail = pu & (np.nan_to_num(arr, nan=-1) >= cut)
    return float(np.nansum(arr[tail & x_full]) / np.nansum(arr[tail]))
s4_dir = RUNS / "A" / S4_ID / "anchor"
S4 = {}
if s4_dir.exists():
    r4 = rep("A", S4_ID); t4 = float(SC["S4_carbon"]["targets"]["irrecoverable_carbon_m_soc"])
    c4 = float(r4["irrecoverable_carbon_m_soc"])
    x4 = np.zeros(pu.shape, bool); x4[pu] = ec.read_selections(s4_dir / "portfolio.tif", pu)[0]
    S4 = dict(target=t4, capture=round(c4, 4), binds=bool(abs(c4 - t4) <= TOL),
              tail_msoc=round(tail_capture(x4, soc), 4), tail_biomass=round(tail_capture(x4, bio), 4))
    print(f"S4 pilot (level A, theta {TH}x): m_soc capture {c4:.4f} vs target {t4:.3f} -> "
          f"{'BINDS at the kink' if S4['binds'] else ('ABOVE target: non-binding -> step up the ladder (pre-registered)' if c4 > t4 else 'BELOW target?! investigate')}")
    print(f"   theta-tail mass capture: m_soc {S4['tail_msoc']:.3f} (band >= 0.75 -> {'PASS' if S4['tail_msoc'] >= 0.75 else 'FAIL'}) | biomass {S4['tail_biomass']:.3f} (reported; AB tail vacuous)")
    x0 = np.zeros(pu.shape, bool); x0[pu] = ec.read_selections(RUNS / "A" / REF / "anchor" / "portfolio.tif", pu)[0]
    print(f"   S0@A reference: m_soc capture {float(rep('A', REF)['irrecoverable_carbon_m_soc']):.4f} | "
          f"tail m_soc {tail_capture(x0, soc):.3f} | tail biomass {tail_capture(x0, bio):.3f}")
else:
    print("S4 pilot not solved (06) -- skipped")

# ---- T1-analog: block captures per anchor beside intended shares ---------------------------------------------
t1 = []
for level in ("A", "B"):
    for fid, scen in ((REF, "S0_balanced"), ("s0_ssp245_theta5", "S0_balanced"), (S4_ID, "S4_carbon")):
        d = RUNS / level / fid / "anchor"
        if not d.exists(): continue
        r = rep(level, fid)
        row = dict(level=level, formulation=fid)
        for b, fs in BLOCKS.items():
            row[f"{b}_capture"] = round(float(sum(r[f] for f in fs)), 3)
        row["gHM"] = round(float(r["human_modification"]), 3)
        row["EFG mean"] = round(float(r[[i for i in r.index if i not in FEAT and i not in ("human_modification", "irrecoverable_carbon_sl_soc")]].mean()), 3)
        t1.append(row)
T1 = pd.DataFrame(t1); print("\nT1-analog (block capture = sum of member captured fractions; intended shares in scenarios_ab_v1.json):")
print(T1.to_string(index=False))

S4 pilot (level A, theta 2.0x): m_soc capture 0.7720 vs target 0.772 -> BINDS at the kink
   theta-tail mass capture: m_soc 0.877 (band >= 0.75 -> PASS) | biomass 1.000 (reported; AB tail vacuous)
   S0@A reference: m_soc capture 0.7552 | tail m_soc 0.850 | tail biomass 0.977

T1-analog (block capture = sum of member captured fractions; intended shares in scenarios_ab_v1.json):
level      formulation  core_habitat_capture  connectivity_capture  carbon_capture  biodiversity_capture   gHM  EFG mean
    A s0_ssp585_theta5                 0.557                 0.993           1.141                 0.862 0.463     0.699
    A s0_ssp245_theta5                 0.576                 0.990           1.143                 0.863 0.463     0.696
    A s4_ssp585_theta2                 0.555                 0.988           1.177                 0.863 0.464     0.692
    B s0_ssp585_theta5                 0.499                 0.861           1.055                 0.741 0.408     0.600
    B s0_ssp24

In [5]:
# ---- freeze the AB-2 record --------------------------------------------------------------------------------
out = dict(created_utc=datetime.now(timezone.utc).isoformat(), verdict_rule=RULE, verdict_rule_hash=RULE_HASH,
           verdicts=V.to_dict(orient="records"), nesting=dict(threshold=NEST_T, primary_level=PRIMARY, **res),
           fg_level_A=fg, s4_pilot=S4, t1_analog=T1.to_dict(orient="records"))
(SPEC / "gate_ab2_verdicts.json").write_text(json.dumps(out, indent=1, default=str))
print(f"wrote spec/gate_ab2_verdicts.json -> primary budget level = {PRIMARY}; H-AB4 = {hab4.verdict}")

wrote spec/gate_ab2_verdicts.json -> primary budget level = B; H-AB4 = PLATEAU-RICH


## → next

Enter R3 (AB-0), R4 (AB-1), R5 (AB-2) in `spec/results_log.md`. The report-back for chat: H-AB4,
the nesting verdict (which level is primary), the S4 pilot (bind / step up), E14-analog. Then
`09_ab3_freeze` builds the 14-formulation manifest at the primary level.